# Growth & Retention Intelligence: Smart Salon Recommendation

This notebook demonstrates the MVP implementation for the Smart Salon Recommendation Engine. Since real user behavioral data is not available, we rely on synthetic data generated via the `generate_data.py` script.

## 1. Load Data
Loading the generated CSV files.

In [ ]:
import csv

def load_csv(filename):
    with open(filename, 'r') as f:
        return list(csv.DictReader(f))

users = load_csv('data/users.csv')
shops = load_csv('data/shops.csv')
bookings = load_csv('data/bookings.csv')

print(f"Loaded {len(users)} users, {len(shops)} shops, {len(bookings)} bookings.")

## 2. Evaluation Logic
We test our API logic (`api.py`) on specific user segments:
1. New User (Cold Start)
2. Repeat Loyal User
3. Budget User
4. Premium User

In [ ]:
import sys
sys.path.append('.')
from api import recommend_shops
import json

def print_res(title, res):
    print(f"\n=== {title} ===")
    print(f"Confidence: {res['confidence']}")
    print(f"Persona inferred: {res.get('debug_persona', 'Cold Start')}")
    for i, shop in enumerate(res['recommendedShops']):
        print(f"{i+1}. Shop {shop['shopId']} | Score: {shop['score']}")

# 1. Cold Start
print_res("Cold Start User", recommend_shops({"lat": 40.71, "lon": -74.00, "preferred_service_type": "Haircut"}))

# 2. Targeted Persona: Search for Premium User
premium_user = next(u for u in users if u['persona'] == 'Premium')
print_res("Premium User Profile", recommend_shops({"user_id": premium_user['user_id']}))

# 3. Targeted Persona: Search for Budget User
budget_user = next(u for u in users if u['persona'] == 'Budget')
print_res("Budget User Profile", recommend_shops({"user_id": budget_user['user_id']}))


## 3. Metrics Recap
By adjusting weights dynamically based on user persona (Budget vs Premium), the recommendation score for `Shop A` might be `0.9` for User 1, but `0.3` for User 2.

This increases Conversion by ensuring the subset visible matches the user's spending profile, reduces Churn by preventing bad experiences like long wait times for "Quick Service" users, and allows cold-start discovery using generic fallbacks.